# Spread souverains et risque politique en zone euro

Ces dernières années, la question de la dette publique et de la santé financière des États est revenue au cœur de l'actualité économique, en particulier au sein de la zone euro. Avec la fin des politiques de taux d'intérêt négatifs, le durcissement de la politique monétaire et les crises budgétaires successives, les marchés financiers scrutent de très près la viabilité des finances publiques. L'accentuation rapide et marquée des écarts de taux s'explique généralement par la combinaison d'une instabilité politique, de déficits budgétaires persistants et d'un niveau élevé d'endettement public. Lorsque ces facteurs se cumulent, les investisseurs craignent une perte de contrôle budgétaire ou un risque de non-soutenabilité, ce qui peut provoquer une réévaluation brutale du risque, une fuite vers des actifs plus sûrs et une envolée des coûts de financement pour les États concernés.

Pour mesurer et comparer la santé financière de ces pays, les économistes et les marchés utilisent un indicateur clé : le spread souverain. Très simplement, le spread correspond à l'écart de taux d'intérêt entre l'obligation d'un État à 10 ans et celle de l'Allemagne, qui sert de référence « sans risque » dans la zone euro. Il se calcule par une simple soustraction :


$$\text{Spread} = \text{Taux du pays} - \text{Taux de l'Allemagne}$$


Si la France emprunte à 3 % et l'Allemagne à 2 %, le spread français est de 1 % (soit 100 points de base). Plus ce chiffre est élevé, plus les marchés estiment que le pays présente un risque, ce qui alourdit le coût de financement de sa dette.

De nombreuses études académiques se sont penchées sur ces dynamiques de marché et sur la formation de ces écarts de taux. Les travaux de **Paul De Grauwe et Yuemei Ji (2012)**, notamment sur la fragilité inhérente de la zone euro, montrent que les spreads ne dépendent pas seulement des fondamentaux économiques d'un pays, mais peuvent aussi fluctuer sous l'effet de mouvements de panique auto-réalisateurs des marchés. D'autres recherches, comme celles de **Kerstin Bernoth, Jürgen von Hagen et Ludger Schuknecht (2012)** sur la tarification du risque souverain, soulignent quant à elles le rôle central de la dette et des déficits dans l'évaluation du risque par les investisseurs. 

Notre projet s'inscrit dans cette lignée : en nous concentrant sur la France, l'Allemagne, l'Italie et l'Espagne à partir de 2016, nous chercherons à observer et analyser l'évolution de ces spreads souverains au fil du temps.


# PARTIE 1 : Collecte et nettoyage des données

## A. Importation des données :

### 0. Installation


In [ ]:
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd
import requests

### 1. Préparation de l'importation
Pour structurer l'analyse des spreads souverains, l'étude se focalise sur quatre pays de la zone euro : la France, l'Allemagne, l'Italie et l'Espagne (`COUNTRIES = ["FR", "DE", "IT", "ES"]`), à partir du 1er janvier 2016 (`START_DATE`).

Nous créons `RAW DIR` afin de stocker les données brutes et éviter des requêtes répétées lors de la relance du script.

Des en-têtes personnalisés (`HEADERS`) incluant un User-Agent sont configurés pour identifier proprement le script auprès des serveurs et éviter d'éventuels blocages, associés à un `TIMEOUT` de 60 secondes.


In [ ]:
# Définition des chemins d'accès aux données (dans spreads_souverains/config.py)
from spreads_souverains.config import COUNTRIES, DATA_DIR, RAW_DIR, START_DATE

# Paramètres des requêtes HTTP pour l'importation des données
HEADERS = {"User-Agent": "Mozilla/5.0 (projet ENSAE - spreads souverains)"}
TIMEOUT = 60

### 2. Origine des données 

Pour récupérer l'historique des taux des quatre pays (France, Allemagne, Italie, Espagne) depuis le 1er janvier 2016, nous centralisons trois flux de données directement en ligne :

- **CNBC** (`CNBC_URL`) : Récupération des taux journaliers (les cours de clôture) via l'API du site, grâce à des codes propres à chaque pays (`CNBC_SYMBOLS`).
- **Bundesbank** (`BUNDESBANK_URL`) : Téléchargement direct des taux allemands à 10 ans, qui servent de taux de référence ("sans risque") pour toute la zone euro.
- **Eurostat** (`EUROSTAT_UR`) : Téléchargement d'un fichier (au format `SDMX-CSV`) contenant les taux mensuels de convergence à long terme de l'Union européenne.

Il est important de noter que CNBC est notre source principale. Bindesbank et Eurostat sont des sources officielles qui nous permettrons de vérifier les données de CNBC.

In [ ]:
# Définition des URLs pour l'importation des données
CNBC_SYMBOLS = {c: f"{c}10Y-{c}" for c in COUNTRIES}
CNBC_URL = (
    "https://ts-api.cnbc.com/harmony/app/bars/{symbol}/1D/"
    "{start}000000/{end}000000/adjusted/EST5EDT.json"
)
BUNDESBANK_URL = (
    "https://api.statistiken.bundesbank.de/rest/download/BBSIS/"
    "D.I.ZST.ZI.EUR.S1311.B.A604.R10XX.R.A.A._Z._Z.A?format=csv&lang=en"
)
EUROSTAT_URL = (
    "https://ec.europa.eu/eurostat/api/dissemination/sdmx/2.1/data/"
    "irt_lt_mcby_m/M.MCBY.{geos}?format=SDMX-CSV&startPeriod={start}"
)

La fonction `_get(url : str)` nous permet de récuperer les fichiers bruts en évitant des bugs réseau.

In [ ]:
def _get(url: str) -> requests.Response:
    response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
    response.raise_for_status()  # Lève une exception si le statut HTTP indique une erreur
    return response

### 3. Données journalières par pays 
La fonction `fetch_cnbc_country` récupére les cours de clôture quotidiens des obligations souveraines à 10 ans pour chaque pays de notre périmètre, en passant par l'API de CNBC.

Cette fonction s'appuie sur notre système de cache local (`RAW_DIR`): si le fichier n'a pas encore été téléchargé, une requête est envoyée à l'API et le fichier brut est stocké au format `JSON`. Le script lit ensuite ce fichier pour en extraire les prix et les dates, avant de structurer le tout sous forme de série temporelle propre et triée chronologiquement.

In [ ]:
def fetch_cnbc_country(country: str, refresh: bool = False) -> pd.Series:
    """Rendement à 10 ans journalier (clôture, en %) d'un pays."""
    cache = RAW_DIR / f"cnbc_{country}10Y.json"
    if refresh or not cache.exists():  # Mécanisme de cache : on ne télécharge que si nécessaire
        url = CNBC_URL.format(
            symbol=CNBC_SYMBOLS[country],
            start=START_DATE.strftime("%Y%m%d"),
            end=date.today().strftime("%Y%m%d"),
        )
        cache.write_text(_get(url).text)

    bars = json.loads(cache.read_text())["barData"]["priceBars"] 
    series = pd.Series(
        {pd.Timestamp(bar["tradeTime"][:8]): float(bar["close"]) for bar in bars},
        name=country,
    ) 
    return series.sort_index()

La fonction `fetch_cnbc` permet de regrouper les séries individuelles de chaque pays dans un tableau unifié.

Pour ce faire, le script utilise la fonction `pd.concat` afin d'aligner côte à côte (en colonnes via l'argument `axis=1`) les séries de taux obtenues pour l'ensemble des pays du périmètre. L'index du tableau final est ensuite explicitement nommé date afin de garantir une structuration homogène et exploitable sous forme de DataFrame.


In [ ]:
def fetch_cnbc(refresh: bool = False) -> pd.DataFrame:
    """Tableau (dates x pays) des rendements à 10 ans CNBC, en %."""
    frame = pd.concat(
        [fetch_cnbc_country(c, refresh=refresh) for c in COUNTRIES], axis=1
    )  # Concaténation des séries de chaque pays en colonnes
    frame.index.name = "date"
    return frame

### 4. Benchmark allemand

La fonction `fetch_bundesbank`récupère les emprunts fédéraux allemands à 10 ans. 

Cette fonction récupère les taux mensuels de long terme d'Eurostat via l'API, puis transforme les données brutes d'un format long à un format large pour que chaque pays apparaisse en colonne.

In [ ]:
def fetch_bundesbank(refresh: bool = False) -> pd.Series:
    """Taux à 10 ans des titres fédéraux allemands (Svensson), en %."""
    cache = RAW_DIR / "bundesbank_10y.csv"
    if refresh or not cache.exists():
        cache.write_bytes(_get(BUNDESBANK_URL).content)

    raw = pd.read_csv(
        cache,
        skiprows=9,
        header=None,
        names=["date", "value", "flag"],
        na_values=["."],
        encoding="utf-8-sig",
    )  # Lecture du CSV en ignorant l'en-tête technique
    raw["date"] = pd.to_datetime(raw["date"], errors="coerce")  # Nettoyage des dates invalides
    series = raw.dropna(subset=["date", "value"]).set_index("date")["value"]
    series = pd.to_numeric(series, errors="coerce").dropna()  # Conversion propre en numérique
    return series.loc[START_DATE:].rename("DE_bundesbank")


### 5. Taux mensuels de convergence de l'Union Européenne 

La fonction `fetch_eurostat_monthly`récupère les taux mensuels de convergence de l'Union Européenne publiés par Eurostat sous forme de moyennes mensuelles (`irt_lt_mcby_m`). 

La fonction récupère les données ia l'API puis les transforme le tableau brut d'un format long à un format large pour que les périodes forment l'index et que les pays apparaissent en colonnes.

In [ ]:
def fetch_eurostat_monthly(refresh: bool = False) -> pd.DataFrame:
    """Taux de convergence à 10 ans (moyennes mensuelles), en %."""
    cache = RAW_DIR / "eurostat_irt_lt_mcby_m.csv"
    if refresh or not cache.exists():
        url = EUROSTAT_URL.format(
            geos="+".join(COUNTRIES), start=START_DATE.strftime("%Y-%m")
        )
        cache.write_bytes(_get(url).content)

    raw = pd.read_csv(cache)
    frame = raw.pivot(index="TIME_PERIOD", columns="geo", values="OBS_VALUE")  # Passage du format long au large
    frame.index = pd.PeriodIndex(frame.index, freq="M")
    frame.index.name = "mois" 
    return frame[COUNTRIES]

### 6. Collecte des données 

Enfin, le bloc principal du script s'assure d'abord que l'arborescence des dossiers de stockage brut (raw) est bien créée (`RAW_DIR.mkdir`), puis déclenche séquentiellement l'exécution des fonctions d'importation. Le paramètre `refresh = False` est configuré par défaut pour exploiter le cache local.

In [ ]:
RAW_DIR.mkdir(parents=True, exist_ok=True)  # Crée l'arborescence des dossiers si elle n'existe pas

refresh = False
fetch_cnbc(refresh)  # "refresh=True" permet de forcer le téléchargement si nécessaire
fetch_bundesbank(refresh)
fetch_eurostat_monthly(refresh)

##  B. Nettoyage des taux souverains à 10 ans (FR, DE, IT, ES)

**Objectif:** Partir des données brutes téléchargées par `import_cleaning/importation_donnees.py` (dossier `data/raw/`) pour obtenir des séries propres, alignées et comparables entre pays, enregistrées dans `data/cleaned/`.

**Démarche,** pour chaque source :
1. on affiche les données brutes
2. on repère ce qui pose problème
3. on corrige
4. on vérifie le résultat

Les fichiers sont lus depuis `data/raw/`, copiés et modifiés par les fonctions de `import_cleaning/cleaning.py` puis la version "propre" se trouve dans `data/cleaned/`.

### 0. Préparation

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

from import_cleaning import cleaning
from spreads_souverains import config

DATA_DIR = config.DATA_DIR
RAW_DIR = config.RAW_DIR
COUNTRIES = cleaning.COUNTRIES
DAY_NAMES = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

plt.rcParams["figure.figsize"] = (11, 4)

print("Fichiers bruts :", [p.name for p in RAW_DIR.iterdir()])
print("Début de la période d'étude :", cleaning.START_DATE.date())

### 1. CNBC : rendements journaliers à 10 ans

#### 1.1 Le fichier brut

L'API CNBC renvoie un fichier JSON par pays. Chaque « barre » correspond à une journée de cotation.

In [ ]:
path = RAW_DIR / "cnbc_FR10Y.json"
bars = json.loads(path.read_text())["barData"]["priceBars"]

print(f"{len(bars)} barres pour la France. Les 3 premières :")
for bar in bars[:3]:
    print(bar)

In [ ]:
for country in COUNTRIES:
    path = RAW_DIR / f"cnbc_{country}10Y.json"
    bars = json.loads(path.read_text())["barData"]["priceBars"]
    first_date = bars[0]["tradeTime"][:8]
    last_date = bars[-1]["tradeTime"][:8]
    print(f"{country} : {len(bars)} barres, du {first_date} au {last_date}")

print("Type du champ 'close' :", type(bars[0]["close"]))

**Dans le fichier brut :**
- `close` est le rendement de clôture en %, mais il est stocké comme texte (`"0.9890"`) : nous le convertirons à un objet `float`.
- `tradeTime` est une date au format `AAAAMMJJhhmmss` : on garde les 8 premiers caractères et on le tansforme en un objet `date`.
- On ignore les colonnes `volume`, `open`, `high` et `low`.
- Les pays n'ont pas le même nombre de barres sur la même période : leurs calendriers ne sont pas alignés, on va compléter les valeurs manquantes par pays.

`cleaning.load_cnbc()` regroupe les quatre pays dans un seul tableau (dates en lignes, pays en colonnes). Cette étape **met en forme** les données, sans encore rien supprimer.

In [ ]:
cnbc_raw = cleaning.load_cnbc()
print("Dimensions :", cnbc_raw.shape)
cnbc_raw.head(10)

In [ ]:
print("Nombre de valeurs manquantes par pays :")
cnbc_raw.isna().sum()

In [ ]:
cnbc_raw.plot(title="CNBC : rendements à 10 ans BRUTS (%) par pays", linewidth=0.8)
plt.ylabel("%")
plt.show()

À l'œil, les courbes semblent cohérentes, mais le tableau contient déjà des `NaN` dès les premières lignes : le 2 et le 3 janvier 2016 sont un samedi et un dimanche.

#### 1.2 Problème n°1 : des cotations le week-end

In [ ]:
# Exemple sur une semaine
week = cnbc_raw.loc["2016-01-07":"2016-01-12"].copy()
week["jour"] = [DAY_NAMES[d] for d in week.index.dayofweek]
week

On remarque qu'il y a certain week-ends avec les valeurs complétées, surtout le samedi, d'autres non. De plus elles n'existent pas pour tous les pays le même jour.

Cependant, les marchés de dette souveraine de la zone euro sont fermés le week-end. Ces valeurs ne sont donc pas de vraies clôtures : ce sont des cotations indicatives et peu liquides. Les garder aurait trois effets :
- Fausses variations : un taux qui bouge un samedi augmente artificiellement la volatilité ;
- Pays décalés : un spread FR − DE calculé un samedi comparerait une cotation hors séance à une valeur manquante ;
- Calendrier irrégulier : une semaine aurait 5, 6 ou 7 observations selon le pays.

Donc, on ne garde que les jours du lundi au vendredi (`dayofweek < 5`), à partir du début de la période d'étude.

In [ ]:
before = cnbc_raw.loc[cleaning.START_DATE:]
step1 = before[before.index.dayofweek < 5]

print(f"Lignes avant : {len(before)}, après : {len(step1)}")
print(f"→ {len(before) - len(step1)} lignes de week-end supprimées")
print("Samedis ou dimanches restants :", (step1.index.dayofweek >= 5).sum())

#### 1.3 Problème n°2 : des jours ouvrés manquants

Quand on met les pays côte à côte, un jour absent pour un pays apparaît comme un `NaN`. Mais un jour absent pour tous les pays disparaît complètement du tableau.

On a donc recréé le calendrier complet des jours ouvrés avec `pd.bdate_range`, puis on aligne le tableau dessus avec `reindex`. Les jours absents deviennent des lignes de `NaN`, visibles.

In [ ]:
all_business_days = pd.bdate_range(step1.index.min(), step1.index.max())
print(f"Jours ouvrés du calendrier : {len(all_business_days)}")
print(f"Jours présents dans les données : {len(step1)}")

absent_for_all = [d.date() for d in all_business_days if d not in step1.index]
print("Jours absents pour TOUS les pays :", absent_for_all)

step2 = step1.reindex(all_business_days)

In [ ]:
incomplete = step2[step2.isna().any(axis=1)].copy()
incomplete["jour"] = [DAY_NAMES[d] for d in incomplete.index.dayofweek]
print(f"{len(incomplete)} jours ouvrés où au moins un pays manque :")
incomplete

In [ ]:
# Longueur des trous : rolling(n).sum() == n signifie "n valeurs manquantes d'affilée"
for country in COUNTRIES:
    missing = step2[country].isna()
    n_two = (missing.rolling(2).sum() == 2).sum()
    n_three = (missing.rolling(3).sum() == 3).sum()
    print(f"{country} : {missing.sum()} jours manquants, {n_two} fois 2 jours d'affilée, {n_three} fois 3 jours d'affilée")

**Diagnostic :** Les trous sont rares et courts (1 ou 2 jours, jamais 3). Ils tombent surtout sur des jours fériés nationaux (Noël et le 26 décembre, lundi de Pâques, 1er mai…).

**Solution :** Nous avons décidé de reporter la dernière valeur (`ffill`). Cela suppose que le taux n'a pas bougé pendant le jour manquant, mais cela reste cohérent avec un marché fermé, et n'utilise que le passé

On limite le report à 2 jours (`ffill(limit=2)`). Au-delà, supposer un taux constant devient trop fort : on préfère alors supprimer la ligne.

**Contrepartie :** un jour comblé a une variation nulle, ce qui sous-estime un tout petit peu la volatilité. L'effet est négligeable vu le petit nombre de jours concernés.


#### 1.5 Résultat

In [ ]:
cnbc, cnbc_filled = cleaning.clean_yields(cnbc_raw)
cleaning.check_daily(cnbc, cnbc_filled, "CNBC 10 ans")
cnbc.describe().round(2)

In [ ]:
cnbc.plot(title="CNBC : rendements à 10 ans NETTOYÉS (%)", linewidth=0.8)
plt.ylabel("%")
plt.show()


On obtient un calendrier régulier de jours ouvrés, sans valeur manquante, où les 4 pays sont observés les mêmes jours. Un spread compare donc toujours deux valeurs du même jour.

### 2. Bundesbank : taux allemand à 10 ans

#### 2.1 Le fichier brut

In [ ]:
lines = (RAW_DIR / "bundesbank_10y.csv").read_text(encoding="utf-8-sig").splitlines()

print("Début du fichier :")
for line in lines[:13]:
    print(line)
print("...")
print("Fin du fichier :")
for line in lines[-3:]:
    print(line)

**Ce qu'on lit dans le fichier brut :**
- les 9 premières lignes sont des métadonnées: on va les sauter avec `skiprows=9` ;
- une valeur manquante est notée `.,No value available` : on la lit comme `NaN` avec `na_values=["."]` ;
- le fichier contient tous les jours depuis 1997, week-ends compris, on va restreindre.

On commence par charger le fichier sous la bonne forme avec `cleaning.load_bundesbank()`.

In [ ]:
bb_raw = cleaning.load_bundesbank()
bb_period = bb_raw.loc[cleaning.START_DATE:]
bb_period.head(10)

Analyse de la localisation des données manquantes :

In [ ]:
share_missing = bb_period["DE_bundesbank"].isna().groupby(bb_period.index.dayofweek).mean()
share_missing.index = DAY_NAMES
print("Part de valeurs manquantes selon le jour :")
share_missing.round(3)

**Diagnostic.** Les samedis et dimanches sont **toujours** vides, et quelques jours ouvrés aussi (jours fériés allemands). C'est le même problème que pour CNBC. On regarde la longueur des trous sur les jours ouvrés :

In [ ]:
bb_bdays = bb_period[bb_period.index.dayofweek < 5]
bb_bdays = bb_bdays.reindex(pd.bdate_range(bb_bdays.index.min(), bb_bdays.index.max()))

missing = bb_bdays["DE_bundesbank"].isna()
three_in_a_row = missing.rolling(3).sum() == 3
print(f"{missing.sum()} jours ouvrés manquants")
print("Trous de 3 jours, qui se terminent le :", [d.date() for d in missing[three_in_a_row].index])

pd.DataFrame({
    "avant": bb_bdays.loc["2018-12-20":"2018-12-28", "DE_bundesbank"],
    "après ffill": bb_bdays.loc["2018-12-20":"2018-12-28", "DE_bundesbank"].ffill(limit=cleaning.FFILL_LIMIT),
})

La Bundesbank ne publie pas les jours fériés allemands, d'où plus de jours à combler que pour CNBC. Quelques années ont un trou de **3 jours** (du 24 au 26 décembre). Avec `ffill(limit=2)`, le 24 et le 25 reçoivent la valeur du 21, mais le 26 reste vide et sa ligne est supprimée : au-delà de 2 jours, on n'invente pas de valeur. Le 1er janvier 2016 est aussi supprimé, car aucune valeur antérieure ne permet de le combler.

#### 2.2 Correction et résultat

On applique **la même fonction `clean_yields`**, avec les mêmes règles, pour que les sources soient comparables.

In [ ]:
bundesbank, bundesbank_filled = cleaning.clean_yields(bb_raw)
cleaning.check_daily(bundesbank, bundesbank_filled, "Bundesbank DE 10 ans")

bundesbank.plot(title="Bundesbank : taux allemand à 10 ans NETTOYÉ (%)", linewidth=0.8, legend=False)
plt.ylabel("%")
plt.show()

### 3. Eurostat : taux de convergence mensuels

#### 3.1 Le fichier brut

In [ ]:
eu_csv = pd.read_csv(RAW_DIR / "eurostat_irt_lt_mcby_m.csv")
print("Dimensions :", eu_csv.shape)
eu_csv.head()

**Ce qu'on lit dans le fichier brut :**
- le fichier a un format particulier : une ligne par couple (pays, mois). Pour comparer les pays, il faut le changer à un format mois en lignes, pays en colonnes : c'est ce que fait `pivot` ;
- plusieurs colonnes ont une seule valeur (`DATAFLOW`, `freq`, `int_rt`…) et n'apportent rien ;
- la donnée est mensuelle, donc pas de problème de week-end. Il reste à vérifier qu'aucun mois ne manque.

#### 3.2 Correction et résultat

In [ ]:
eurostat = cleaning.clean_monthly(cleaning.load_eurostat())
cleaning.check_monthly(eurostat, "Eurostat 10 ans (mensuel)")
eurostat.head()

In [ ]:
eurostat_plot = eurostat.copy()
eurostat_plot.index = pd.to_datetime(eurostat_plot.index)  # "2016-01" -> date, pour le graphique
eurostat_plot.plot(title="Eurostat : taux de convergence à 10 ans (%)")
plt.ylabel("%")
plt.show()

### 5. Enregistrement dans `data/cleaned/`

Les trois fichiers sont écrits par `cleaning.save_cleaned(cnbc, bundesbank, eurostat)`, appelée par `main.py` (`python3 main.py` dans un terminal). On les relit ici.

In [ ]:
print("\nRelecture des fichiers enregistrés :")
for path in config.CLEANED_DIR.glob("*.csv"):
    df = pd.read_csv(path, index_col=0)
    print(f"- {path.name} : {df.shape[0]} lignes, {df.shape[1]} colonnes, de {df.index[0]} à {df.index[-1]}")

**Limites :**
- Le calendrier est celui du lundi au vendredi : un jour férié où une source publie quand même une valeur (par exemple le 1er janvier) est conservé.
- Avec `ffill(limit=2)`, un trou de 3 jours ou plus voit ses 2 premiers jours comblés et les suivants supprimés. Les trous de CNBC ne dépassent pas 2 jours ; côté Bundesbank, cela ne concerne que quelques 26 décembre.
- Les jours comblés ont une variation nulle, ce qui sous-estime légèrement la volatilité.
- CNBC n'est pas une source officielle : il faut désormais s'occuper de la comparaison entre ses sources pour vérifier sa fiabilité.

### 6. Fiabilité de CNBC : comparaison avec Eurostat et la Bundesbank

CNBC n'étant pas une source officielle, on le compare à deux sources qui le sont :
- **Eurostat** (mensuel, 4 pays) pour vérifier le **niveau** des taux ;
- **Bundesbank** (journalier, Allemagne) pour vérifier les **variations au jour le jour**, qui comptent le plus dans notre étude d'événements.

In [ ]:
# CNBC en moyenne mensuelle, comparé à Eurostat
eurostat_dates = eurostat.copy()
eurostat_dates.index = pd.to_datetime(eurostat_dates.index)  # "2016-01" -> 2016-01-01
cnbc_monthly = cnbc.resample("MS").mean()                     # un point par mois, daté du 1er

gap_monthly = (cnbc_monthly - eurostat_dates).dropna() * 100  # écart en points de base
print("Écart CNBC - Eurostat (pb) :")
display(pd.DataFrame({
    "ecart_moyen_pb": gap_monthly.mean(),
    "ecart_abs_moyen_pb": gap_monthly.abs().mean(),
    "ecart_abs_max_pb": gap_monthly.abs().max(),
}).round(2))

fig, axes = plt.subplots(2, 2, figsize=(12, 7), sharex=True)
for ax, country in zip(axes.flatten(), COUNTRIES):
    ax.plot(cnbc_monthly[country], label="CNBC (moyenne mensuelle)")
    ax.plot(eurostat_dates[country], color="black", linestyle="--", linewidth=1, label="Eurostat")
    ax.set_title(country)
    ax.set_ylabel("Taux à 10 ans (%)")
    ax.grid(alpha=0.3)
    ax.legend()
fig.suptitle("CNBC et Eurostat, moyennes mensuelles")
fig.tight_layout()
plt.show()

Les courbes se superposent : l'écart absolu moyen est de 1 à 4 pb selon le pays, très faible devant des spreads de plusieurs dizaines de pb. Pour l'Allemagne, l'écart (≈ 4 pb) est presque toujours de même signe : c'est un décalage constant (obligation de référence différente), qui déplace le niveau des spreads mais pas leurs variations.

In [ ]:
# Taux allemand à 10 ans : CNBC et Bundesbank, jour par jour
daily = pd.DataFrame({"cnbc": cnbc["DE"], "bundesbank": bundesbank["DE_bundesbank"]}).dropna()

print("Corrélation CNBC - Bundesbank :")
print("- niveaux :", round(daily["cnbc"].corr(daily["bundesbank"]), 3))
for step in [1, 2]:
    changes = daily.diff(step).dropna()
    print(f"- variations à {step} jour(s) :", round(changes["cnbc"].corr(changes["bundesbank"]), 3))

gap_daily = (daily["cnbc"] - daily["bundesbank"]) * 100
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(gap_daily, linewidth=0.8)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Écart CNBC - Bundesbank sur le taux allemand à 10 ans")
ax.set_ylabel("Points de base")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

Les niveaux sont quasi identiques (corrélation ≈ 0,999). Les variations à 1 jour sont moins corrélées (≈ 0,48), mais la corrélation monte à 2 jours (≈ 0,73) : cela traduit surtout un décalage d'horaire (clôture de marché pour CNBC, courbe estimée par la Bundesbank) plutôt que des erreurs. **On retient CNBC comme source principale**, et une fenêtre de réaction de 2 jours autour des événements sera plus robuste qu'une fenêtre de 1 jour.

## B. Importation des données pour avoir une base de titres d'articles politiques 
**Objectif :** constituer une base de titres d'articles politiques français sur environ dix ans, à laquelle on appliquera ensuite un algorithme d'analyse de sentiments. On n'a pas besoin du texte complet : le titre, la date et le lien suffisent.
Nous avons explicité les les sources testées, les raisons des abandons, puis la méthode retenue (Media Cloud), avec le code commenté.
Nous avons décidé de prendre les articles de **Franceinfo**, en supposant une neutralité politique de leur part. 

### Essai n°1 : fichiers bruts GDELT en local (abandonné)
GDELT publie aussi ses données en libre téléchargement : un fichier toutes les 15 minutes (96 par jour). Cependant, les fichiers ne sont pas triés par média. Pour garder seulement France Info, il faut télécharger tous les articles du monde puis filtrer : environ 2 900 fichiers pour un mois et 350 000 pour dix ans. C'est beaucoup trop lourd. À titre d'idée, pour un seul jour de données, nous devions télécharger un fichier de plus de 2Go.

### Essai n°3 : API DOC de GDELT (abandonné)
Avec l'API DOC, le filtrage est fait chez GDELT : on demande « les titres de tel média, sur tels thèmes, tel jour ». Mais nous étions confrontés à deux restrictions : 250 articles au maximum par demande, et pas plus d'une demande toutes les 5 secondes.
Or, on a l'impression que GDELT a bloqué notre connexion car on recevait le message (« Please limit requests to one every 5 seconds »), même avec 30 secondes d'attente et même depuis le navigateur.
On est donc passés à Media Cloud.

### Solution retenue : Media Cloud
Media Cloud est une archive de presse en ligne gérée par des universitaires, avec une bibliothèque Python officielle (`mediacloud`). Par rapport à GDELT :
- Pas de limite de 250 articles : on récupère tous les résultats, page par page.
- La bibliothèque gère seule le rythme des demandes, donc on n'a plus de blocage.
- On peut cibler un média précis grâce à son numéro dans l'annuaire Media Cloud.
- **Pré-requis :** Nous avons du créer un compte gratuit sur [search.mediacloud.org](https://search.mediacloud.org), afin de récupérer une clé API dans notre profil.

#### Réglages

- Période : de 2016 à aujourd'hui.
- Définition d'un article « politique » : Media Cloud ne classe pas les articles par thème. On garde donc ceux qui contiennent au moins un mot d'une liste de vocabulaire politique. `OR` signifie « ou », et les guillemets servent pour les expressions de plusieurs mots. Le mot est cherché dans tout le texte de l'article, pas seulement dans le titre, donc la sélection est large.

In [ ]:
import datetime as dt
import os

import mediacloud.api
import pandas as pd
from dotenv import load_dotenv

from spreads_souverains.config import ROOT, TITRES_BRUTS_FILE

# La clé est lue dans le fichier .env (non versionné), voir .env.example
load_dotenv(ROOT / ".env")
CLE_API = os.getenv("MEDIACLOUD_API_KEY")

# Période (année, mois, jour).
DATE_DEBUT = dt.date(2016, 1, 1)
DATE_FIN = dt.date.today()

# Mots-clés de recherche pour les articles à récupérer
RECHERCHE = ('gouvernement OR "Premier ministre" OR Assemblée OR Sénat OR élection '
             'OR élections OR Élysée OR "motion de censure" OR dissolution OR législatives '
             'OR présidentielle OR ministre OR député OR députés OR "49.3" OR budget')

# France Info a changé d'adresse le 14 mai 2025 : Media Cloud les voit comme deux médias
NOMS_FRANCEINFO = ["francetvinfo.fr", "franceinfo.fr"]

FICHIER_SORTIE = TITRES_BRUTS_FILE


def connexion_mediacloud(cle_api=CLE_API):
    """Renvoie l'annuaire des médias et l'outil de recherche d'articles."""
    annuaire = mediacloud.api.DirectoryApi(cle_api)
    recherche = mediacloud.api.SearchApi(cle_api)
    return annuaire, recherche


annuaire, recherche = connexion_mediacloud()

####  Trouver France Info
Le diagnostic a fait apparaître deux difficultés :

1. Plusieurs médias portent un nom proche. Une recherche sur `francetvinfo.fr` renvoie aussi `mobile.francetvinfo.fr`, qui n'a que 6 articles. Notre premier essai a pris ce sous-site par erreur et n'a trouvé que 5 articles politiques en dix ans. On garde donc uniquement le média dont le nom est*exactement celui du site. Le site principal (numéro 107654) contient environ 307 000 articles.
2. France Info a changé d'adresse le 14 mai 2025 : `francetvinfo.fr` est devenu `franceinfo.fr`. Pour Media Cloud, ce sont deux médias distincts. Avec la seule ancienne adresse, on obtenait 0 article à partir de juin 2025. On cherche donc sur les deux adresses.


In [ ]:
def trouver_ids_franceinfo(annuaire, noms=NOMS_FRANCEINFO):
    """Renvoie les numéros Media Cloud des médias dont le nom est exactement dans noms."""
    ids = []
    for nom in noms:
        resultats = annuaire.source_list(name=nom, platform="online_news", limit=50)["results"]
        for source in resultats:
            if source["name"] == nom:          # nom exact : on écarte mobile.francetvinfo.fr, etc.
                ids.append(source["id"])
                print("Trouvé :", nom, "-> numéro", source["id"])
    return ids


IDS_FRANCEINFO = trouver_ids_franceinfo(annuaire)

#### Importation des titres

Les choix les plus importants :
- Pagination : Media Cloud renvoie les articles par pages de 1 000. Chaque réponse contient un jeton (`pagination_token`) qui indique la page suivante. Quand le jeton vaut `None`, on a atteint la dernière page.
- Mois par mois : à la fin de chaque mois, on ajoute les titres au CSV (`mode="a"`). L'importation dure très longtemps (plusieurs heures), donc cela permettait de l'arrêter et la reprendre plus tard en changeant `DATE_DEBUT`, sans rien perdre.

In [ ]:
def collecter_titres(recherche, source_ids, date_debut=DATE_DEBUT, date_fin=DATE_FIN,
                     fichier_sortie=FICHIER_SORTIE):
    """Ajoute au CSV les titres (date, titre) de chaque mois de la période."""
    os.makedirs(os.path.dirname(fichier_sortie), exist_ok=True)
    debuts_de_mois = pd.date_range(start=date_debut.replace(day=1), end=date_fin, freq="MS")
    print("Nombre de mois à traiter :", len(debuts_de_mois))

    for debut_mois in debuts_de_mois:
        # Premier et dernier jour du mois (sans sortir de la période choisie)
        debut = max(debut_mois.date(), date_debut)
        fin = min((debut_mois + pd.offsets.MonthEnd(0)).date(), date_fin)

        articles_du_mois = []
        jeton_page = None          # le "jeton" indique à Media Cloud quelle page donner ensuite

        while True:
            try:
                page, jeton_page = recherche.story_list(
                    RECHERCHE, debut, fin,
                    source_ids=source_ids,
                    page_size=1000,
                    pagination_token=jeton_page,
                )
            except Exception as erreur:
                print("   Erreur pour ce mois :", erreur)
                break

            articles_du_mois = articles_du_mois + page

            # S'il n'y a plus de jeton, c'est qu'on a eu la dernière page
            if jeton_page is None:
                break

        print(debut, "->", fin, ":", len(articles_du_mois), "articles")

        # Ajouter les articles du mois au fichier CSV
        if len(articles_du_mois) > 0:
            tableau = pd.DataFrame(articles_du_mois)
            tableau = tableau[["publish_date", "title"]]
            tableau.columns = ["date", "titre"]

            # On écrit les noms de colonnes seulement si le fichier n'existe pas encore
            deja_la = os.path.exists(fichier_sortie)
            tableau.to_csv(fichier_sortie, mode="a", header=not deja_la,
                           index=False, encoding="utf-8")


collecter_titres(recherche, IDS_FRANCEINFO)

#### Nettoyage du fichier final

Comme on ajoute les titres mois par mois, une reprise de la collecte peut écrire deux fois le même article. On enlève les doublons (même date et même titre), on trie par date, puis on affiche le nombre d'articles par année.

In [ ]:
def dedoublonner_titres(fichier_sortie=FICHIER_SORTIE):
    """Enlève les doublons (même date et même titre), trie par date et réécrit le CSV."""
    if not os.path.exists(fichier_sortie):
        print("Aucun article récupéré.")
        return None

    tableau = pd.read_csv(fichier_sortie, encoding="utf-8-sig")
    tableau["date"] = pd.to_datetime(tableau["date"])
    tableau = tableau.drop_duplicates(subset=["date", "titre"])
    tableau = tableau.sort_values("date", kind="stable")
    tableau.to_csv(fichier_sortie, index=False, encoding="utf-8-sig")
    return tableau


def afficher_statistiques(tableau):
    """Affiche le nombre total d'articles et le nombre d'articles par année."""
    print()
    print("Nombre total d'articles :", len(tableau))
    print()
    print("Nombre d'articles par année :")
    print(tableau["date"].dt.year.value_counts().sort_index().to_string())


titres = dedoublonner_titres()
if titres is not None:
    afficher_statistiques(titres)

## B.bis. Recherche des plus gros évènements politiques.
### 1. Idée de la méthode

Quand un événement politique majeur survient (dissolution, motion de censure, chute d'un gouvernement), France Info publie beaucoup plus d'articles politiques que d'habitude. On repère donc les jours où le volume d'articles s'écarte fortement de son niveau normal.

Étapes :
1. compter les articles politiques par jour ;
2. calculer un niveau « habituel » pour chaque jour ;
3. repérer les pics (jours bien au-dessus du niveau habituel) ;
4. regrouper les pics proches en un seul événement ;
5. décrire chaque événement par ses mots-clés et quelques titres.

### 2. Réglages

In [8]:
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re

FICHIER_TITRES = "titres_franceinfo_mediacloud.csv"
FICHIER_EVENTS = "events.csv"

FENETRE = 30            # nombre de jours pour calculer le niveau habituel
SEUIL = 2.0             # un pic = au moins 2 fois plus d'articles que d'habitude
MINIMUM_ARTICLES = 20   # évite les faux pics les jours où l'archive est presque vide
ECART_MAX = 2           # deux pics à moins de 2 jours = un seul événement

### 3. Premier essai : beaucoup de bruit

Le premier essai, sur la base brute, a donné ce classement (extrait) :

| Rang | Date | Mots-clés |
|---|---|---|
| 1 | 2018-01-19 | 2018, janvier, vendredi, 19.01.2018, jeudi, 1ère |
| 2 | 2018-01-23 | 2018, janvier, 23.01.2018, mardi, 1ère, partie |
| 4 | 2025-09-09 | lecornu, sébastien, chute, bayrou, devant, septembre |
| 5 | 2020-03-15 | municipales, résultats, élections, tour, coronavirus |
| 9 | 2026-03-09 | moyen, orient, guerre, 2026, mars, lundi |
| 15 | 2026-06-03 | juin, mercredi, 2026, garros, roland |

On a identifié trois problèmes :
1. Des vidéos de journaux télévisés (« …vendredi 19.01.2018, 1ère partie ») gonflent artificiellement certains jours, et les dates remontent en mots-clés.
2. Des sujets non politiques (Moyen-Orient, Roland-Garros, incendies) : la recherche Media Cloud cherche les mots dans *tout le texte* de l'article, donc un article sur un incendie qui cite « le ministre » est compté comme politique.
3. Des événements attendus manquent (dissolution de juin 2024, censure de Barnier en décembre 2024) : ils sont noyés dans ce bruit.

Certains vrais événements ressortent malgré tout (chute de Bayrou et nomination de Lecornu en septembre 2025, municipales de mars 2020), ce qui montre que la méthode fonctionne une fois le bruit retiré.

### 4. Correction : deux filtres avant de compter

**Filtre 1 : on enlève les vidéos de JT**, reconnaissables à une date du type `19.01.2018` ou aux mentions « 1ère partie », « replay ».

**Filtre 2 : on garde seulement les titres politiques**, c'est-à-dire ceux dont le titre lui-même contient un mot de la liste `MOTS_POLITIQUES` (institutions, élections, Premiers ministres, partis…).



In [9]:
MOTS_POLITIQUES = [
    "gouvernement", "premier ministre", "ministre", "matignon", "élysée",
    "assemblée", "sénat", "député", "députés", "sénateur", "parlement",
    "motion de censure", "censure", "dissolution", "49.3", "remaniement", "démission",
    "élection", "élections", "législatives", "présidentielle", "municipales",
    "européennes", "régionales", "scrutin", "sondage", "référendum",
    "budget", "réforme", "projet de loi", "conseil constitutionnel",
    "macron", "philippe", "castex", "borne", "attal", "barnier", "bayrou", "lecornu",
    "le pen", "bardella", "mélenchon", "rassemblement national", "rn", "lfi",
    "insoumis", "républicains", "socialiste", "renaissance", "nupes",
    "nouveau front populaire", "nfp", "gilets jaunes", "retraites",
]
MOTIF_VIDEOS = r"\d{1,2}\.\d{1,2}\.\d{4}|1ère partie|2e partie|2ème partie|replay"

base = pd.read_csv(FICHIER_TITRES, encoding="utf-8-sig")
base["date"] = pd.to_datetime(base["date"]).dt.normalize()   # on garde le jour seulement
base = base.dropna(subset=["titre"])
print("Titres chargés :", len(base))

# Filtre 1 : vidéos de JT
base = base[~base["titre"].str.contains(MOTIF_VIDEOS, case=False, regex=True)]
print("Après suppression des vidéos de JT :", len(base))

# Filtre 2 : titres politiques
motif_politique = r"\b(?:" + "|".join(re.escape(m) for m in MOTS_POLITIQUES) + r")\b"
base = base[base["titre"].str.contains(motif_politique, case=False, regex=True)]
print("Titres politiques gardés :", len(base))

FileNotFoundError: [Errno 2] No such file or directory: 'titres_franceinfo_mediacloud.csv'

### 5. Repérer les pics

- On complète le calendrier avec les jours sans article (valeur 0).
- Niveau habituel est défini comme la médiane des 30 jours précédents 
- Ratio = articles du jour / niveau habituel.

In [ ]:
par_jour = base.groupby("date").size()
calendrier = pd.date_range(par_jour.index.min(), par_jour.index.max(), freq="D")
jours = pd.DataFrame({"nb_articles": par_jour.reindex(calendrier, fill_value=0)})

jours["habituel"] = jours["nb_articles"].shift(1).rolling(FENETRE, min_periods=10).median()
jours["ratio"] = jours["nb_articles"] / (jours["habituel"] + 1)
jours["est_un_pic"] = (jours["ratio"] >= SEUIL) & (jours["nb_articles"] >= MINIMUM_ARTICLES)

pics = jours[jours["est_un_pic"]]
print("Jours de pic :", len(pics))

### 6. Regrouper les pics en événements

Une crise peut durer plusieurs jours (par exemple, l'annonce de la dissolution puis les réactions du lendemain). On change de numéro d'événement dès que l'écart avec le pic précédent dépasse `ECART_MAX` jours : `cumsum()` additionne ces « ruptures » pour numéroter les groupes.

In [ ]:
ecart = pics.index.to_series().diff().dt.days
pics = pics.assign(evenement=(ecart > ECART_MAX).cumsum().values)

### 7. Décrire chaque événement

Un mot est **caractéristique** d'un jour s'il y est beaucoup plus fréquent que d'habitude : on divise le nombre de titres du jour qui le contiennent par sa fréquence moyenne sur toute la base. On écarte les mots vides, les jours, les mois et les nombres (sauf « 49.3 »).

On choisit ensuite comme exemples les 3 titres qui contiennent le plus de ces mots-clés.

In [ ]:
MOTS_VIDES = set('''
le la les un une des du de d l au aux et ou en dans sur pour par avec sans
ce cet cette ces son sa ses leur leurs qui que quoi dont il elle ils elles on
nous vous je tu se ne pas plus moins est sont a ont été être avoir fait faire
comme mais aussi tout tous toute toutes après avant entre contre vers chez
quand comment pourquoi selon face lors depuis encore déjà très bien
direct vidéo info infos franceinfo entretien replay partie 1ère
lundi mardi mercredi jeudi vendredi samedi dimanche
janvier février mars avril mai juin juillet août septembre octobre novembre décembre
'''.split())

def decouper_en_mots(titre):
    mots = [m.strip(".") for m in re.findall(r"[a-zàâäéèêëîïôöùûüç0-9\.]+", titre.lower())]
    return [m for m in mots
            if len(m) >= 4 and m not in MOTS_VIDES
            and (not any(c.isdigit() for c in m) or m == "49.3")]

compteur_global = Counter()
for titre in base["titre"]:
    compteur_global.update(set(decouper_en_mots(titre)))
nb_jours_total = base["date"].nunique()

def mots_caracteristiques(titres_du_jour, combien=6):
    compteur_jour = Counter()
    for titre in titres_du_jour:
        compteur_jour.update(set(decouper_en_mots(titre)))
    scores = {mot: nb / (compteur_global[mot] / nb_jours_total + 1)
              for mot, nb in compteur_jour.items() if nb >= 3}
    return sorted(scores, key=scores.get, reverse=True)[:combien]

def titres_representatifs(titres_du_jour, mots_cles, combien=3):
    titres = pd.Series(titres_du_jour.unique())
    score = titres.apply(lambda t: len(set(decouper_en_mots(t)) & set(mots_cles)))
    return " | ".join(titres[score.sort_values(ascending=False).index].head(combien))

### 8. Construire `events.csv`

Pour chaque événement, on retient le jour le plus intense (`date_pic`), les dates de début et de fin, le volume d'articles, le niveau habituel, le ratio, les mots-clés et des exemples de titres. Les événements sont classés du plus intense au moins intense.

In [ ]:
lignes = []
for numero, groupe in pics.groupby("evenement"):
    jour_max = groupe["nb_articles"].idxmax()
    titres_du_jour = base.loc[base["date"] == jour_max, "titre"]
    mots_cles = mots_caracteristiques(titres_du_jour)
    lignes.append({
        "date_pic": jour_max.date(),
        "date_debut": groupe.index.min().date(),
        "date_fin": groupe.index.max().date(),
        "nb_articles_pic": groupe["nb_articles"].max(),
        "niveau_habituel": round(groupe.loc[jour_max, "habituel"], 1),
        "ratio": round(groupe["ratio"].max(), 2),
        "mots_cles": ", ".join(mots_cles),
        "exemples_titres": titres_representatifs(titres_du_jour, mots_cles),
    })

events = pd.DataFrame(lignes).sort_values("ratio", ascending=False).reset_index(drop=True)
events.insert(0, "rang", events.index + 1)
events.to_csv(FICHIER_EVENTS, index=False, encoding="utf-8-sig")

events[["rang", "date_pic", "nb_articles_pic", "ratio", "mots_cles"]].head(15)

### 9. Contrôle visuel

La courbe du nombre d'articles politiques par jour, avec un point rouge sur chaque événement détecté, permet de vérifier d'un coup d'œil que les pics retenus sont bien des ruptures nettes.

In [ ]:
plt.figure(figsize=(14, 5))
plt.plot(jours.index, jours["nb_articles"], linewidth=0.6, color="grey",
         label="Articles politiques par jour")
plt.scatter(pd.to_datetime(events["date_pic"]), events["nb_articles_pic"],
            color="red", s=15, label="Événements détectés")
plt.title("Articles politiques France Info par jour et événements détectés")
plt.ylabel("Nombre d'articles")
plt.legend()
plt.tight_layout()
plt.savefig("events_graphique.png", dpi=150)
plt.show()

### 10. Validation et limites

**Validation :** la méthode doit retrouver des événements connus, par exemple la dissolution (9 juin 2024), la censure du gouvernement Barnier (4 décembre 2024) et la chute du gouvernement Bayrou (8 septembre 2025). Sur une base de test construite avec ces trois événements, des pics parasites de vidéos de JT et d'incendies, les filtres écartent le bruit et les trois événements sont retrouvés aux bonnes dates.

**Limites :**
- Le volume mesure l'attention médiatique, pas la gravité : un sujet très couvert mais sans effet sur les marchés peut ressortir.
- La liste `MOTS_POLITIQUES` est faite à la main et peut être .
- La médiane glissante s'adapte lentement : pendant une crise longue (plusieurs semaines), le niveau habituel monte et les pics suivants sont moins bien détectés.


## C. Date de réaction du marché aux événements

`scripts/detect_ev.py` repère les événements politiques comme des pics d'articles et les enregistre dans `outputs/tables/events.csv`, chacun daté par son jour de pic (`date_pic`). Ce jour n'est pas toujours un jour de marché : un pic un samedi ou un dimanche ne peut se voir dans les taux que le lundi suivant.

`add_reaction_dates` (dans `spreads_souverains/events.py`) ajoute donc à chaque événement sa date de réaction : la première séance de notre calendrier CNBC nettoyé à partir du jour du pic.

In [ ]:
from spreads_souverains.events import add_reaction_dates, load_events

events = load_events()
events = add_reaction_dates(events, cnbc.index)

events_week_end = events[events["date_pic"].dt.dayofweek >= 5]
print(f"{len(events)} événements, dont {len(events_week_end)} avec un pic le week-end")
events[["rang", "date_pic", "date_reaction", "nb_articles_pic", "mots_cles"]].head(10)

## D. Statistiques descriptives des spreads

**Objectif :** décrire l'évolution des taux et des spreads depuis 2016, puis la comparer avant et après la dissolution de l'Assemblée nationale (rupture au 10 juin 2024, premier jour de marché après l'annonce du 9 juin au soir).

Nous partons du panel `data/yields.csv` : les taux à 10 ans des quatre pays en %, et les spreads `s_FR`, `s_IT`, `s_ES` contre le Bund allemand, en points de base (1 pb = 0,01 point de pourcentage). Les fonctions sont dans `spreads_souverains/descriptives.py`.

In [ ]:
from spreads_souverains import descriptives

panel = descriptives.charger_panel()
print("Période :", panel.index.min().date(), "->", panel.index.max().date(), "|", len(panel), "séances")
panel.tail()

### 1. Niveaux des taux et des spreads

In [ ]:
descriptives.tracer_rendements(panel)
descriptives.tracer_spreads(panel);

Les quatre taux suivent la même tendance : proches de zéro, voire négatifs pour l'Allemagne, jusqu'en 2021, puis en forte hausse à partir de 2022 avec le resserrement monétaire de la BCE. Les spreads, eux, enlèvent ce mouvement commun.

Le spread français reste longtemps le plus bas des trois, entre 20 et 81 pb (maximum en février 2017). Il fait un saut net au moment de la dissolution : 49 pb le vendredi 7 juin 2024, 56 pb le lundi 10 juin et 82 pb le vendredi 14 juin. À l'inverse, les spreads italien et espagnol baissent sur la même période.

### 2. Comparaison avant et après la dissolution

Nous découpons la période en deux sous-périodes (`PERIODS` dans `config.py`). Pour chacune, nous calculons le spread moyen et l'écart-type des variations quotidiennes, qui mesure la volatilité.

In [ ]:
tableau_periodes = descriptives.statistiques_par_periode(panel)

Le spread français moyen passe de 40 à 77 pb, alors que les spreads italien (de 171 à 98 pb) et espagnol (de 98 à 59 pb) baissent. Sa volatilité augmente aussi (de 1,8 à 2,6 pb), quand celles de l'Italie et de l'Espagne diminuent.

**Limite :** la période « avant » mélange des régimes très différents (taux négatifs, crise italienne de 2018, Covid en 2020, hausse des taux en 2022). Une moyenne sur huit ans et demi peut masquer des évolutions internes ; la figure 9 montre le détail dans le temps.

### 3. La France face à l'Espagne et à l'Italie

L'écart France - Espagne vaut `s_FR - s_ES` : le taux allemand s'annule, il reste la différence entre les taux français et espagnol. Un écart positif signifie que la France emprunte plus cher que l'Espagne.

In [ ]:
descriptives.tracer_ecarts(panel);

De 2016 à 2023, la France emprunte toujours moins cher que l'Espagne : l'écart reste entre -124 et -26 pb, sans un seul jour positif. Il se resserre à partir de 2024, s'accélère après la dissolution et devient positif pour la première fois le 26 septembre 2024. La France passe ensuite au-dessus de l'Italie (premier jour le 7 octobre 2025). À la dernière séance disponible, la France paie environ 92 pb de plus que l'Espagne et 43 pb de plus que l'Italie.

### 4. Volatilité et corrélation glissantes

Les statistiques par période résument huit ans en un chiffre. Pour suivre leur évolution, nous les calculons sur une fenêtre glissante de 60 séances (environ trois mois) : l'écart-type des variations quotidiennes de chaque spread, et la corrélation des variations du spread français avec celles des spreads italien et espagnol.

In [ ]:
descriptives.tracer_volatilite_correlation(panel);

La volatilité du spread français reste faible jusqu'en 2024, (1,5 pb en médiane de 2016 à 2023), hormis trois pics autour de 4 pb : au printemps 2017, pendant la campagne présidentielle, au printemps 2020 (Covid) et début 2023. Elle monte à la dissolution, puis atteint son niveau le plus élevé de la période fin 2026 (5,3 pb sur les 60 dernières séances).

Les variations du spread français sont plus corrélées à celles des autres pays après la dissolution : de 0,51 à 0,74 avec l'Italie et de 0,59 à 0,72 avec l'Espagne, sur l'ensemble de chaque période. Une forte corrélation indique que les spreads bougent ensemble au jour le jour ; elle ne dit pas lequel entraîne l'autre, et un mouvement commun peut coexister avec un écart de niveau qui se creuse, comme le montre la figure 8.

**Limite :** les jours fériés sont comblés en recopiant la veille lors du nettoyage, ce qui crée des variations nulles et sous-estime légèrement la volatilité.

## E. Étude d'événements

Pour chaque événement de `events.csv`, nous mesurons la réaction des spreads français, italien et espagnol à partir de sa date de réaction J0 (première séance après le pic d'articles) :

- $\Delta s_{J0} = s(J0) - s(J-1)$ : la variation du spread le jour de réaction ;
- $\Delta s_{[-1;+5]} = s(J+5) - s(J-2)$ : la variation cumulée de la séance J-1 à la séance J+5 ;
- $z = \Delta s_{J0} / \sigma_{60}$, où $\sigma_{60}$ est l'écart-type des variations quotidiennes sur les 60 séances précédant J0.

Le z-score compare la variation du jour à la volatilité habituelle du moment : $|z| > 2$ signale une variation inhabituelle.

Nous faisons le même calcul pour l'Italie et l'Espagne : si leurs spreads bougent autant que le spread français le jour d'un événement, la variation vient probablement d'un mouvement commun à la zone euro plutôt que de la politique française.

### 1. Mesures par événement

In [ ]:
from spreads_souverains.events import add_reaction_dates, load_events

events = add_reaction_dates(load_events(), panel.index)

for pays in ["FR", "IT", "ES"]:
    s = panel[f"s_{pays}"]
    panel[f"var_J0_{pays}"] = s.diff()                                    # s(J0) - s(J-1)
    panel[f"var_fenetre_{pays}"] = s.shift(-5) - s.shift(2)               # s(J+5) - s(J-2)
    panel[f"volatilite_60_{pays}"] = s.diff().rolling(60).std().shift(1)  # 60 séances avant J0
    panel[f"z_{pays}"] = panel[f"var_J0_{pays}"] / panel[f"volatilite_60_{pays}"]

# On garde seulement les jours d'événement
tableau = panel.loc[events["date_reaction"]].copy()
tableau["mots_cles"] = events["mots_cles"].values
tableau[["mots_cles", "var_J0_FR", "var_J0_IT", "var_J0_ES", "z_FR", "z_IT", "z_ES"]].round(2)

### 2. Moyenne et médiane sur l'ensemble des événements

Les événements font monter ou baisser les spreads : en moyenne, les hausses et les baisses se compensent. Nous calculons donc la moyenne et la médiane des variations signées (le sens moyen de la réaction), puis de leurs valeurs absolues (l'ampleur de la réaction, quel que soit son sens).

In [ ]:
colonnes = ["var_J0_FR", "var_J0_IT", "var_J0_ES",
            "var_fenetre_FR", "var_fenetre_IT", "var_fenetre_ES",
            "z_FR", "z_IT", "z_ES"]

signe = tableau[colonnes].agg(["mean", "median"])
absolu = tableau[colonnes].abs().agg(["mean", "median"])

resume = pd.concat([signe, absolu], keys=["signé", "valeur absolue"]).T
resume.round(2)

Sur les 49 événements, il n'y a pas de sens moyen : les variations signées sont proches de zéro, donc un événement politique ne fait pas monter le spread de façon systématique. C'est pour cette raison qu'il faut aussi regarder les valeurs absolues.

La réaction française est un peu plus forte que celle des autres pays, mais l'écart est faible. En |z| moyen, la France est à 1,04, contre 0,83 pour l'Italie et 0,91 pour l'Espagne : le spread français réagit donc un peu plus, rapporté à sa volatilité habituelle. Un |z| moyen autour de 1 signifie cependant qu'un jour d'événement ressemble à une journée à peine plus agitée que la normale.

Les variations en points de base sont trompeuses. Le spread italien bouge davantage en pb (4,14 contre 2,05 pour la France), mais c'est parce qu'il est beaucoup plus volatil en général. Le z-score corrige ce biais : c'est donc lui qu'il faut comparer entre pays.

Ces moyennes ne suffisent pas à conclure à un effet des événements politiques : il faut encore les comparer aux jours sans événement.

Pour compléter ces moyennes, nous regardons comment les spreads évoluent autour des événements. Pour chaque événement, nous suivons les spreads français, italien et espagnol de J-5 à J+10, en écart à leur valeur de la veille J-1 : toutes les trajectoires passent donc par 0 en J-1, ce qui permet de les comparer. Nous faisons ensuite la moyenne sur les 49 événements. Si le spread français s'écarte des deux autres après J0, la réaction est propre à la France ; s'ils bougent ensemble, il s'agit plutôt d'un mouvement commun à la zone euro.

In [ ]:
from spreads_souverains import config
from spreads_souverains.controles import save_figure

jours = range(-5, 11)

fig, ax = plt.subplots(figsize=(10, 5))
for pays in ["FR", "IT", "ES"]:
    trajectoires = []
    for date in events["date_reaction"]:
        i = panel.index.get_loc(date)                       # position de J0 dans le panel
        s = panel[f"s_{pays}"].iloc[i - 5 : i + 11].values  # spread de J-5 à J+10
        trajectoires.append(s - panel[f"s_{pays}"].iloc[i - 1])
    trajectoires = pd.DataFrame(trajectoires, columns=jours)
    ax.plot(jours, trajectoires.mean(), color=config.COLORS[pays], linewidth=2, marker="o", label=f"{pays} - DE")

ax.axvline(0, color="black", linestyle=":", linewidth=1)
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xticks(list(jours))
ax.set_xlabel("Séances autour de J0")
ax.set_ylabel("Écart au spread de J-1 (pb)")
ax.set_title("Figure 10 : trajectoire moyenne des spreads de J-5 à J+10 (49 événements)")
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()
save_figure(fig, "10_trajectoire_moyenne")

Les trois spreads bougent très peu autour de J0, moins de 1 pb. Ensuite, les spreads italien et espagnol baissent nettement (environ -4 et -2,5 pb à J+10), alors que le spread français reste stable, autour de -0,5 pb : après un événement politique français, la France ne profite donc pas de la détente observée chez ses voisins. Ces écarts restent petits et reposent sur des moyennes de 49 événements ; ce n'est qu'une piste, à confirmer par la comparaison avec les jours sans événement.

### 3. Jours d'événement et autres jours

Les moyennes de la partie précédente n'ont de sens que comparées à une référence. Nous comparons donc l'ampleur des variations les 49 jours d'événement à celle des quelque 2 800 autres séances :

- le **ratio** $|\Delta s|$ moyen les jours d'événement / $|\Delta s|$ moyen les autres jours, en pb puis en $|z|$. Un ratio de 1 signifie qu'un jour d'événement ressemble à un jour ordinaire ;
- le **percentile** : on classe les $|\Delta s|$ de toutes les séances de la plus petite à la plus grande, et on regarde où tombent les jours d'événement. Sans effet, le percentile médian vaut environ 50 et 10 % des jours d'événement tombent dans les 10 % de séances les plus agitées.

In [ ]:
# True les jours de réaction à un événement, False les autres séances
jour_evenement = panel.index.isin(events["date_reaction"])
print(jour_evenement.sum(), "jours d'événement sur", len(panel), "séances")

lignes = []
for pays in ["FR", "IT", "ES"]:
    variation = panel[f"var_J0_{pays}"].abs()   # |Δs| de chaque séance (pb)
    z = panel[f"z_{pays}"].abs()                 # |z| de chaque séance
    percentile = variation.rank(pct=True) * 100  # rang de |Δs| parmi toutes les séances (0 à 100)
    lignes.append({
        "spread": f"{pays} - DE",
        "abs_var_evenement": variation[jour_evenement].mean(),
        "abs_var_autres": variation[~jour_evenement].mean(),
        "ratio_var": variation[jour_evenement].mean() / variation[~jour_evenement].mean(),
        "abs_z_evenement": z[jour_evenement].mean(),
        "abs_z_autres": z[~jour_evenement].mean(),
        "ratio_z": z[jour_evenement].mean() / z[~jour_evenement].mean(),
        "percentile_median": percentile[jour_evenement].median(),
        "part_top_10pct": (percentile[jour_evenement] > 90).mean() * 100,
    })

comparaison = pd.DataFrame(lignes).set_index("spread")
comparaison.round(2)

Les jours d'événement sont plus agités que les autres pour les trois spreads, mais nettement plus pour la France. Le spread français varie en moyenne de 2,05 pb un jour d'événement contre 1,19 pb un autre jour, soit un ratio de 1,73, contre 1,28 pour l'Italie et 1,42 pour l'Espagne.

Une partie de cet écart vient du calendrier : une quinzaine d'événements tombent après 2024, une période où le spread français est plus volatil en général. Le ratio en $|z|$, qui rapporte chaque variation à la volatilité des 60 séances précédentes, corrige ce biais. Il reste plus élevé pour la France (1,43) que pour l'Italie (1,07) et l'Espagne (1,21).

Le percentile confirme ce résultat. Le jour d'événement médian se classe au 73e percentile des variations françaises, contre 52 pour l'Italie et 54 pour l'Espagne, qui restent proches de ce qu'on attendrait sans effet (50). Un quart des jours d'événement (24 %) se situe dans les 10 % de séances les plus agitées pour la France, contre 10 % attendus sans effet.

### 4. Les plus fortes variations tombent-elles sur un événement ?

Lors du contrôle des données (étape 13), nous avons listé les 10 plus fortes variations quotidiennes de chaque spread. Nous comptons maintenant combien d'entre elles suivent un événement, avec la même règle : un pic d'articles le jour même ou dans les 3 jours précédents, pour qu'un événement du week-end soit rattaché au lundi.

In [ ]:
# Même règle qu'à l'étape 13 : un événement est retenu si son pic d'articles
# tombe dans les 3 jours qui précèdent la variation (ou le jour même)
def evenements_proches(date):
    return events[(events["date_pic"] >= date - pd.Timedelta(days=3)) & (events["date_pic"] <= date)]

for pays in ["FR", "IT", "ES"]:
    variation = panel[f"var_J0_{pays}"].dropna()
    top_dates = variation.abs().sort_values(ascending=False).head(10).index
    nb_avec_evenement = 0
    print(f"\nSpread {pays} : 10 plus fortes variations quotidiennes")
    for d in top_dates:
        proches = evenements_proches(d)
        if len(proches) > 0:
            nb_avec_evenement += 1
        mots = " / ".join(proches["mots_cles"]) if len(proches) > 0 else "-"
        print(f"- {d.date()} : {variation[d]:+.1f} pb | {mots}")
    print(f"=> {nb_avec_evenement} sur 10 tombent sur un événement")

# Point de comparaison : combien en attendrait-on par hasard ?
part_proche = sum(len(evenements_proches(d)) > 0 for d in panel.index) / len(panel)
print(f"\nPart des séances avec un événement dans les 3 jours : {part_proche:.1%}")
print(f"Nombre attendu par hasard dans un top 10 : {10 * part_proche:.1f}")

Aucune des 10 plus fortes variations du spread français ne tombe sur un événement détecté, ni aucune de celles de l'Italie. L'Espagne en compte 2, mais pas pour des raisons propres à la politique espagnole :
- le 24 avril 2017, lendemain du premier tour de la présidentielle française, les trois spreads baissent fortement. Le recul du risque de sortie de l'euro profite à toute la périphérie, et l'Espagne (-17 pb) gagne davantage que la France (-12,6 pb) ;
- le 17 mars 2020, l'événement est le premier tour des municipales, mais la séance est surtout marquée par le krach du Covid.

Par hasard, on attendrait environ 0,6 événement dans un top 10, puisque 5,7 % des séances suivent un événement de près. Le résultat n'est donc pas inférieur au hasard, mais il montre que **les grands chocs de spread ne viennent pas des pics d'actualité politique**. Ce sont plutôt des chocs européens ou mondiaux : la décision de la BCE du 18 mars 2020 (programme d'achats d'urgence PEPP), visible le 19 mars sur les trois spreads, le Brexit le 24 juin 2016, la crise politique italienne de mai 2018.

**Limite :** la méthode de détection rate certains événements. La dissolution du 9 juin 2024 n'est pas dans `events.csv`, alors que la semaine du 10 au 14 juin compte l'une des 10 plus fortes hausses du spread français (+12,7 pb le 14 juin). Pendant une crise longue, le nombre d'articles reste élevé plusieurs jours de suite : le niveau habituel augmente et le pic ne se détache plus.

### 5. Contrôle par l'Espagne et l'Italie

Un spread contre le Bund peut bouger pour des raisons qui n'ont rien à voir avec la France : une décision de la BCE, une tension sur toute la périphérie ou un mouvement de fuite vers l'actif sûr allemand. Sans contrôle, un tel choc peut passer pour une réaction à la politique française.

L'Espagne et l'Italie servent de contrôle, puisqu'elles subissent les mêmes chocs européens que la France mais pas sa politique intérieure. Nous calculons donc la variation des écarts de taux :

$$\Delta(s_{FR} - s_{ES}) = \Delta s_{FR} - \Delta s_{ES}$$

Le taux allemand s'annule : il ne reste que la part de la variation française que l'Espagne ne connaît pas. Si un événement est vraiment français, cet écart doit bouger plus que d'habitude. Nous reprenons les mêmes indicateurs que dans la partie 3.

In [ ]:
# Écarts de la France avec l'Espagne et l'Italie : le taux allemand s'annule
# Δ(s_FR - s_ES) = Δs_FR - Δs_ES : la part de la variation française que l'Espagne ne connaît pas
for autre in ["ES", "IT"]:
    ecart = panel["s_FR"] - panel[f"s_{autre}"]
    panel[f"var_J0_FR_{autre}"] = ecart.diff()
    panel[f"z_FR_{autre}"] = ecart.diff() / ecart.diff().rolling(60).std().shift(1)

lignes = []
for nom in ["FR", "FR_ES", "FR_IT"]:
    variation = panel[f"var_J0_{nom}"].abs()
    z = panel[f"z_{nom}"].abs()
    percentile = variation.rank(pct=True) * 100
    lignes.append({
        "serie": nom,
        "abs_var_evenement": variation[jour_evenement].mean(),
        "abs_var_autres": variation[~jour_evenement].mean(),
        "ratio_var": variation[jour_evenement].mean() / variation[~jour_evenement].mean(),
        "abs_z_evenement": z[jour_evenement].mean(),
        "abs_z_autres": z[~jour_evenement].mean(),
        "ratio_z": z[jour_evenement].mean() / z[~jour_evenement].mean(),
        "percentile_median": percentile[jour_evenement].median(),
        "part_top_10pct": (percentile[jour_evenement] > 90).mean() * 100,
    })

controle = pd.DataFrame(lignes).set_index("serie")
controle.round(2)

In [ ]:
# Figure 11 : ratio |Δ| jours d'événement / autres jours, en pb et en z-score
series = ["FR", "IT", "ES", "FR_ES", "FR_IT"]
noms = ["FR - DE", "IT - DE", "ES - DE", "FR - ES", "FR - IT"]
ratios = pd.concat([comparaison[["ratio_var", "ratio_z"]], controle.loc[["FR_ES", "FR_IT"], ["ratio_var", "ratio_z"]]])

fig, ax = plt.subplots(figsize=(9, 4))
positions = range(len(series))
ax.bar([p - 0.2 for p in positions], ratios["ratio_var"], width=0.4, color="#2a78d6", label="|Δs| en pb")
ax.bar([p + 0.2 for p in positions], ratios["ratio_z"], width=0.4, color="#9cc3ee", label="|z|")
ax.axhline(1, color="black", linewidth=1, linestyle="--")   # 1 = pas de différence
ax.set_xticks(list(positions))
ax.set_xticklabels(noms)
ax.set_ylabel("Jours d'événement / autres jours")
ax.set_title("Figure 11 : réaction les jours d'événement, rapportée aux autres jours")
ax.grid(alpha=0.3, axis="y")
ax.legend()
fig.tight_layout()
save_figure(fig, "11_ratio_evenements")

Une fois le mouvement commun retiré, l'effet des événements se réduit nettement. Le ratio en pb passe de 1,73 pour le spread français contre le Bund à 1,24 pour l'écart France - Espagne et 1,19 pour l'écart France - Italie. En $|z|$, il passe de 1,43 à 1,18 et 1,09. Le percentile médian retombe à 47 et 51, c'est-à-dire au niveau d'une séance ordinaire.

Une bonne partie de la réaction du spread français les jours d'événement est donc un mouvement que partagent l'Italie et l'Espagne. C'est ce que montre aussi le sens des variations : pour 37 événements sur 49, les trois spreads bougent dans le même sens.

In [ ]:
# Les trois spreads vont-ils dans le même sens le jour de l'événement ?
jours_ev = panel.loc[events["date_reaction"]].copy()
jours_ev["mots_cles"] = events["mots_cles"].values
meme_sens = (jours_ev["var_J0_FR"] * jours_ev["var_J0_IT"] > 0) & (jours_ev["var_J0_FR"] * jours_ev["var_J0_ES"] > 0)
print(f"Les trois spreads bougent dans le même sens pour {meme_sens.sum()} événements sur {len(jours_ev)}")

# Les 10 événements où la France s'écarte le plus de l'Espagne
ordre = jours_ev["var_J0_FR_ES"].abs().sort_values(ascending=False).index
colonnes = ["mots_cles", "var_J0_FR", "var_J0_IT", "var_J0_ES", "var_J0_FR_ES", "var_J0_FR_IT"]
jours_ev.loc[ordre, colonnes].head(10).round(1)

Le classement des événements par $|\Delta(s_{FR} - s_{ES})|$ permet de séparer les cas :

- **Des chocs européens.** Le 16 mars 2020 (municipales), les spreads italien (+17 pb) et espagnol (+13 pb) bondissent avec le Covid, bien plus que le français. Le 13 juin 2022 (législatives), la hausse italienne et espagnole (+13 et +9 pb) suit l'annonce par la BCE de la fin de ses achats nets d'obligations et de ses premières hausses de taux. Dans les deux cas, l'écart avec la France est grand, mais la cause n'est pas la politique française.
- **Des réactions propres à la France.** Le 22 février 2017, François Bayrou se rallie à Emmanuel Macron : le spread français baisse de 5 pb alors que l'espagnol monte de 8 pb. À la chute du gouvernement Bayrou (9 septembre 2025) et à la démission de Sébastien Lecornu (6 octobre 2025), le spread français monte de 4 à 5 pb quand les spreads italien et espagnol bougent à peine.
- **Des événements français qui font bouger toute la zone.** Le 24 avril 2017, les trois spreads baissent ensemble. L'écart France - Espagne donne alors une image incomplète, car il retire aussi l'effet de l'événement français sur ses voisins. Le contrôle par l'Espagne et l'Italie est donc prudent : il peut sous-estimer l'effet des événements qui changent le risque de toute la zone euro.

### 6. Bilan de l'étude d'événements

- Les jours d'événement politique sont plus agités que les autres pour le spread français (ratio de 1,4 en $|z|$, 73e percentile médian), et davantage que pour l'Italie et l'Espagne.
- Les 10 plus fortes variations du spread français ne tombent jamais sur un événement détecté : les grands chocs viennent de la BCE, du Covid ou de crises dans d'autres pays.
- Une fois le mouvement commun retiré par l'écart avec l'Espagne ou l'Italie, l'effet moyen devient faible (ratio de 1,1 à 1,2 en $|z|$). Quelques événements restent clairement français, en particulier les crises gouvernementales de 2025.

Ces résultats reposent sur 49 événements et sur des moyennes : ils décrivent des tendances, sans test statistique. La détection par pics d'articles rate aussi des événements majeurs comme la dissolution de 2024.

## F. Indice d'actualité politique et variations du spread français

L'étude d'événements regarde quelques dizaines de dates. Ici, nous utilisons **toutes** les séances : pour chaque jour de marché, nous mesurons l'intensité de l'actualité politique dans les titres France Info, puis nous regardons si le spread français bouge davantage les jours où elle est forte. C'est la « corrélation news-spreads » de notre question.

**Construction de l'indice.** Nous partons des titres politiques de `data/cleaned/titres_franceinfo_politiques.csv`. Un titre compte dans l'indice s'il contient l'un des mots qui signalent une crise gouvernementale : « censure » (et ses dérivés « censuré », « censurer »), « 49.3 », « dissolution », « Premier ministre » ou « Première ministre ». L'indice d'une séance est le **nombre de ces titres**. Chaque titre est rattaché à la première séance de marché à partir de sa date : les titres du samedi et du dimanche comptent pour le lundi, comme pour la date de réaction des événements.

Nous le comparons à l'ampleur des variations quotidiennes, en valeur absolue puisque nous cherchons des mouvements, pas un sens :
- $|\Delta s_{FR}|$, en pb, et $|z_{FR}|$, la variation rapportée à l'écart-type des 60 séances précédentes (comme dans la partie E) ;
- $|\Delta s_{IT}|$ et $|\Delta s_{ES}|$, qui ne devraient pas réagir à la politique française ;
- $|\Delta(s_{FR} - s_{ES})|$ et son $|z|$, la part de la variation française que l'Espagne ne connaît pas.

Les fonctions sont dans `spreads_souverains/indice_articles.py`.

### 1. L'indice

In [ ]:
from spreads_souverains import descriptives, indice_articles

panel = descriptives.charger_panel()
titres = indice_articles.charger_titres()
indice = indice_articles.construire_indice(titres, panel.index)

print(f"{len(indice)} séances, du {indice.index[0].date()} au {indice.index[-1].date()}")
print(f"{indice['nb_mots'].sum()} titres dans l'indice ; au moins un titre lors de {(indice['nb_mots'] > 0).mean():.0%} des séances")
print("Moyenne de l'indice par année :", indice["nb_mots"].groupby(indice.index.year).mean().round(2).to_dict())
indice.sort_values("nb_mots", ascending=False).head(10)

L'indice couvre 2 825 séances, du 4 janvier 2016 au 30 octobre 2026. 1 765 titres contiennent l'un des mots, et 37 % des séances en comptent au moins un. La plupart des jours, il vaut donc 0 ou 1.

Ses plus fortes valeurs correspondent bien à des crises gouvernementales : le recours au 49.3 sur les retraites (annoncé le samedi 29 février 2020, compté le lundi 2 mars), la nomination d'Édouard Philippe (15 mai 2017), la censure du gouvernement Barnier (4 décembre 2024), le 49.3 sur la loi Travail (10 mai 2016), la démission de Sébastien Lecornu (6 octobre 2025), les nominations de Michel Barnier et de François Bayrou (5 septembre et 13 décembre 2024). Le lendemain de la dissolution, le 10 juin 2024, l'indice vaut 8. Contrairement à la détection de pics (partie B.bis), qui compare chaque jour aux 30 précédents, l'indice compte les titres directement : il ne rate donc pas les grands événements d'une période déjà agitée.

**Limites :**
- La liste de mots est courte et faite à la main. Elle attrape parfois l'actualité étrangère : le 24 octobre 2022, plusieurs titres parlent du « futur Premier ministre britannique » Rishi Sunak.
- L'indice moyen passe d'environ 0,4 titre par séance en 2016-2018 à 1,0 à 1,2 en 2024-2026. Une partie de cette hausse reflète la crise politique, une autre la taille de l'archive Media Cloud, plus fournie ces dernières années.

### 2. L'indice et les variations dans le temps

In [ ]:
variations = indice_articles.variations_absolues(panel)
indice_articles.tracer_indice(indice, variations);

Avant 2024, les deux courbes n'ont pas grand-chose en commun. Les périodes où le spread français est le plus agité (début 2017 avec la campagne présidentielle, printemps 2020 avec le Covid) ne correspondent pas à des hausses nettes de l'indice, et le pic de l'indice de mars 2020 (49.3 sur les retraites) est noyé dans le choc du Covid.

Après la dissolution, l'indice et l'ampleur des variations du spread français montent ensemble. Ce mouvement commun est un piège : sur toute la période, il suffit à créer une corrélation, même si les deux séries ne bougent pas ensemble au jour le jour. Nous calculons donc aussi les corrélations séparément avant et après la dissolution, et avec le $|z|$, qui retire l'effet du niveau général de volatilité. En revanche, l'écart France - Espagne ne s'agite pas davantage après 2024 : la volatilité supplémentaire du spread français est en grande partie partagée avec l'Espagne.

### 3. Corrélations

Pour chaque variation, nous calculons la corrélation avec l'indice le jour même (J) et la séance suivante (J+1), car une annonce faite après la clôture des marchés ne peut se voir que le lendemain. La corrélation de Pearson mesure un lien linéaire ; celle de Spearman compare les rangs et résiste mieux aux quelques journées extrêmes.

À titre indicatif, une corrélation est significativement différente de zéro (au seuil de 5 %) si elle dépasse environ $2/\sqrt{n}$ en valeur absolue : 0,04 sur toute la période ($n$ = 2 825 séances), 0,04 avant la dissolution ($n$ = 2 200) et 0,08 après ($n$ = 625).

In [ ]:
from spreads_souverains import config

correlations = indice_articles.tableau_correlations(indice, variations)
indice_articles.enregistrer_indice(indice, correlations)
correlations

In [ ]:
# Mêmes corrélations, avant et après la dissolution (config.PERIODS)
par_periode = {}
for nom, (debut, fin) in config.PERIODS.items():
    par_periode[nom] = indice_articles.tableau_correlations(indice, variations, debut, fin)
pd.concat(par_periode, axis=1).loc[["var_FR", "z_FR", "var_FR_ES", "z_FR_ES"]]

**Sur toute la période**, la corrélation entre l'indice et $|\Delta s_{FR}|$ est de 0,06 le jour même et de 0,03 le lendemain (0,04 et 0,01 avec Spearman). C'est à peine au-dessus du seuil de significativité, et très faible : l'indice n'explique qu'une part infime des variations du spread français. Pour l'Italie, l'Espagne et l'écart France - Espagne en pb, les corrélations sont même légèrement négatives (de -0,03 à -0,08) ; en $|z|$, l'écart France - Espagne est sans lien avec l'indice (0,00).

**Avant la dissolution**, toutes les corrélations sont nulles (entre -0,05 et 0,01) : de 2016 à mi-2024, les jours de forte actualité politique ne sont pas des jours plus agités pour le spread français.

**Après la dissolution**, la corrélation avec le spread français devient positive le jour même : 0,10 en pb et 0,13 en $|z|$, au-dessus du seuil de 0,08. Mais elle reste faible, elle tombe à 0,08 et 0,05 avec Spearman, et elle disparaît le lendemain. Le lien repose donc sur quelques journées très marquées plutôt que sur une relation régulière. Pour l'écart France - Espagne, il ne reste presque rien (0,06 et 0,04 avec Pearson, négatif avec Spearman).

Le décalage d'une séance n'améliore pas le résultat en moyenne, même s'il compte pour certains événements : la censure de Michel Barnier est votée le 4 décembre 2024 vers 20 h 20, après la clôture. Le spread français bouge peu ce jour-là ($|z|$ = 0,3), puis nettement le lendemain ($|z|$ = 3,4).

### 4. L'ampleur des variations selon l'intensité de l'actualité

Une corrélation mesure un lien qui vaut pour tous les jours. Or l'indice vaut 0 ou 1 la plupart du temps. Nous regroupons donc les séances selon la valeur de l'indice et comparons le $|z|$ moyen de chaque groupe.

In [ ]:
par_classe = indice_articles.variations_par_classe(indice, variations)
indice_articles.tracer_classes(par_classe)
par_classe

Les séances avec 0, 1 ou 2 titres ont la même agitation, autour de 0,7 en $|z|$, pour le spread français comme pour l'écart France - Espagne. Le $|z|$ du spread français monte ensuite à 0,91 les séances de 3 à 5 titres et à 1,16 au-delà de 6 titres. Le lien est donc **non linéaire** : seules les journées de très forte actualité politique, environ 5 % des séances (147 sur 2 825), se distinguent. Une corrélation calculée sur toutes les séances le dilue.

Une partie de cet excès est commune à la zone euro : pour l'écart France - Espagne, le $|z|$ ne passe que de 0,74 à 0,87. Et la classe « 6 et plus » ne compte que 29 séances, si bien que quelques jours pèsent beaucoup. Le lendemain de la dissolution (10 juin 2024, $|z|$ = 7,2), le 49.3 du gouvernement Barnier (2 décembre 2024, $|z|$ = 5,0) et la démission de Sébastien Lecornu (6 octobre 2025, $|z|$ = 2,6, et 3,1 pour l'écart avec l'Espagne) font partie des plus fortes réactions. À l'inverse, les nominations de Michel Barnier et de François Bayrou, très commentées, laissent le spread presque immobile ($|z|$ de 0,3 et 0,6).

### 5. Bilan

- Sur l'ensemble des séances, l'actualité politique mesurée par les titres France Info est presque sans lien avec l'ampleur des variations du spread français : corrélation de 0,06 le jour même, nulle le lendemain.
- Ce faible lien n'apparaît qu'**après la dissolution** (0,10 à 0,13 le jour même) : avant juin 2024, les crises gouvernementales françaises ne faisaient pas bouger le spread.
- Il est **non linéaire** : seules les journées de très forte actualité (6 titres ou plus) sont nettement plus agitées, et une partie de ce surcroît est partagée avec l'Espagne.

Ces résultats rejoignent ceux de l'étude d'événements : le risque politique français se voit dans quelques journées de crise depuis 2024, pas dans une relation régulière entre actualité et spreads.

**Limites.** Une corrélation n'est pas une causalité : le marché peut anticiper une crise avant qu'elle n'occupe les titres, ou réagir à une même cause (la situation budgétaire) que la presse. L'indice repose sur cinq mots, sur un seul média et sur une archive dont la taille varie dans le temps. Enfin, il compte les titres par jour calendaire, sans connaître l'heure de publication, ce qui brouille la frontière entre J et J+1.